# EEG–Image–Text conditional tri-modal: strict REAL training-subset pilot

Training data: 200 previously explored concepts from THINGS-EEG2 official **training** partition, sub-01. No official test EEG/image values are used. Preparation deserializes a legacy caption cache that also contains test-caption embeddings, but selects only training caption rows; this is not fully test-blind. This is a development engineering/scientific experiment, not a SOTA result. Uses matched true-caption, pseudo-semantic and cross-concept-shuffled A/B/C/D controls. After a Colab restart, rerun setup and training cells; the stable Drive output path resumes only when config, manifest, source hashes and environment match.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch, pathlib
print('torch',torch.__version__,'GPU',torch.cuda.is_available())


In [ ]:
import os, subprocess, pathlib
REPO='/content/eeg-alignment-phase1'
BRANCH='research/trimodal-baseline-transfer-20261120'
if not pathlib.Path(REPO).exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/firozehrazavi/eeg-alignment-phase1.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',REPO,'checkout',BRANCH],check=True)
actual=subprocess.check_output(['git','-C',REPO,'branch','--show-current'],text=True).strip()
assert actual==BRANCH, f'Expected {BRANCH}, got {actual}'
subprocess.run(['git','-C',REPO,'log','-1','--oneline'],check=True)


In [ ]:
from pathlib import Path
import os, shutil, sys, json
ROOT=Path('/content/drive/MyDrive/eeg_results/patch_phase1/pilot_sub01_200concepts_v1')
CACHE=Path('/content/drive/MyDrive/eeg_results/cache')
SOURCE=Path('/content/EEG20261120_source')
TARGET=Path('/content/EEG20261120_manifest')
SOURCE.mkdir(exist_ok=True)
need={name:ROOT/name for name in ['dataset.json','preparation_state.json','catalog.json','sub-01_train_prez.npy']+[f'train_L{x}_mean.npy' for x in [20,24,28,32,36]]}
need.update({'captions_blip_large.json':CACHE/'captions_blip_large.json','feats7_blip_caps_clip_b32.pt':CACHE/'feats7_blip_caps_clip_b32.pt'})
for name,src in need.items():
    if not src.is_file(): raise FileNotFoundError('Drive file is missing: '+str(src))
    link=SOURCE/name
    if not link.exists():link.symlink_to(src)
    assert link.resolve()==src.resolve()
print('All',len(need),'training-source files located. No test file referenced.')


In [ ]:
import subprocess,sys
if not (TARGET/'train.json').is_file():
    if TARGET.exists(): raise RuntimeError('Incomplete existing training manifest directory; inspect it before removing it')
    subprocess.run([sys.executable,str(Path(REPO)/'prepare_real_dev.py'),'--source',str(SOURCE),'--output',str(TARGET)],check=True)
else:
    print('Reusing the existing manifest; the training loader will verify artifact hashes.')
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_joint_caption_duplicates.py','-v'],check=True,cwd=REPO)


In [ ]:
import json,torch
from pathlib import Path
cfg=json.loads((Path(REPO)/'configs/joint_realdev_cpu.json').read_text())
cfg['device']='cuda' if torch.cuda.is_available() else 'cpu'
cfg_path=Path('/content/EEG20261120_run_config.json')
cfg_path.write_text(json.dumps(cfg,indent=2))
OUT=Path('/content/drive/MyDrive/eeg_results/conditional_trimodal_realdev_resumable_v1')
print('Persistent resumable output:',OUT,'device',cfg['device'])


In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','phase1.joint_experiment','--manifest',str(TARGET/'train.json'), '--config',str(cfg_path),'--output',str(OUT),'--resume'],cwd=REPO,check=True)
print((OUT/'results.csv').read_text())


In [ ]:
# Direct EEG -> Caption score verification WITHOUT any image gallery.
import json, numpy as np, torch, subprocess, sys
from pathlib import Path
cat=json.loads((TARGET/'train.json').read_text())['catalog']
from phase1.joint_data import split_three
_,_,hold=split_three(cat['concept_ids'],cfg['split_seed'])
np.save('/content/EEG20261120_hold_eeg.npy',np.load(TARGET/'sub-01_train_prez.npy',mmap_mode='r')[hold])
np.save('/content/EEG20261120_hold_text.npy',np.load(TARGET/'captions_train_selected.npy',mmap_mode='r')[hold])
score_path=Path('/content/EEG20261120_direct_text_scores.npz')
if score_path.exists(): score_path.unlink()
subprocess.run([sys.executable,'-m','phase1.joint_inference','--checkpoint',str(OUT/'sub-01_s0_B_best.pt'),
 '--eeg','/content/EEG20261120_hold_eeg.npy','--gallery','/content/EEG20261120_hold_text.npy',
 '--modality','text','--output',str(score_path),'--device',cfg['device']],cwd=REPO,check=True)
with np.load(score_path) as v:direct=v['scores']
reference=torch.load(OUT/'sub-01_s0_B_holdout_scores.pt',map_location='cpu',weights_only=True)['text_scores'].numpy()
difference=float(np.max(np.abs(direct-reference)))
assert direct.shape==(200,200) and difference<1e-5
print('PASS image-gallery-free EEG->caption inference, max_abs_score_delta=',difference)


### Scientific boundary
This run uses only previously examined *training* concepts and is not a new official test. Never compare its 200-way development accuracy with historical 06b official 200-way or RPA reported 95.4%. Do not tune the locked development holdout. The repository verifies 06b architecture/state-dict compatibility, but contains no trained historical checkpoint. Before a publication claim: clean concept/image splits, 10 subjects, seeds, matched backbone capacity/compute, caption collision and semantic evaluation, and official evaluation rules.